# MethodThinker 阿里云PAI训练 Notebook

> 全量数据训练 (13,340样本) | QLoRA模式 | 阿里云GPU优化

## 适用平台
- 阿里云PAI-DSW
- 阿里云PAI-EAS
- 阿里云ECS GPU实例

## 绑定目录
- **数据盘**: `/mnt/data` (持久化存储)
- **训练结果**: `/mnt/data/models/{日期}/`

## 训练配置
- **模型**: Qwen/Qwen2.5-Math-1.5B
- **数据**: 13,340样本
- **模式**: QLoRA (4-bit量化 + LoRA)
- **预计时间**: 2-4小时 (V100/A10)

## 1. 环境检查

In [ ]:
# 检查GPU
import torch
print("="*50)
print("阿里云GPU环境检查")
print("="*50)
if torch.cuda.is_available():
    print(f"✓ GPU型号: {torch.cuda.get_device_name(0)}")
    print(f"✓ 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"✓ CUDA版本: {torch.version.cuda}")
    print(f"✓ 支持BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("✗ 未检测到GPU!")

In [ ]:
# 设置HuggingFace镜像（国内加速）
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
print("✓ 已设置HuggingFace镜像")

## 2. 项目准备

In [ ]:
# 克隆项目（如果还没有）
#!git clone https://github.com/alasong/method-thinker.git
#%cd method-thinker
#print("✓ 项目已克隆")

In [ ]:
# 安装依赖
!pip install -q transformers>=4.40.0 accelerate>=0.25.0 peft>=0.7.0 trl>=0.7.0 datasets>=2.14.0 bitsandbytes>=0.41.0
print("✓ 依赖安装完成")

## 3. 训练参数优化

根据GPU显存调整参数：

In [ ]:
# 根据GPU显存选择配置
import torch

gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:  # A100 40GB+
    config = {
        "batch_size": 16,
        "max_length": 4096,
        "gradient_accumulation": 2,
        "lora_r": 64,
        "samples": 13340  # 全量数据
    }
elif gpu_memory >= 24:  # A10/RTX 3090/4090
    config = {
        "batch_size": 8,
        "max_length": 2048,
        "gradient_accumulation": 4,
        "lora_r": 32,
        "samples": 10000  # 使用部分数据
    }
else:  # T4 16GB
    config = {
        "batch_size": 4,
        "max_length": 1024,
        "gradient_accumulation": 8,
        "lora_r": 16,
        "samples": 5000  # 使用部分数据
    }

print("自动选择的训练配置:")
for k, v in config.items():
    print(f"  {k}: {v}")

## 4. 开始训练

In [ ]:
# 创建按日期命名的输出目录
import torch
from datetime import datetime
import os

# 切换到项目根目录
project_root = "/mnt/workspace/method-thinker"
if os.path.exists(project_root):
    os.chdir(project_root)
    print(f"✓ 工作目录: {os.getcwd()}")
else:
    print(f"✗ 项目目录不存在: {project_root}")

# 获取当前日期
today = datetime.now().strftime("%Y-%m-%d")
output_dir = f"/mnt/data/models/{today}"

# 创建目录
os.makedirs(output_dir, exist_ok=True)
print(f"✓ 输出目录: {output_dir}")

# 检测GPU配置
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_memory >= 40:
    # 全量数据训练 (A100 40GB+)
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 3 \
        --batch-size 16 \
        --max-length 4096 \
        --output-dir {output_dir}
elif gpu_memory >= 24:
    # 中等配置 (A10/RTX 3090/4090)
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 2 \
        --batch-size 8 \
        --max-length 2048 \
        --output-dir {output_dir}
else:
    # 低显存配置 (T4 16GB)
    !python scripts/train_sft.py \
        --train-data data/train_data/all_merged.json \
        --use-lora \
        --epochs 1 \
        --batch-size 4 \
        --max-length 1024 \
        --output-dir {output_dir}

print(f"\n✓ 训练完成，模型保存在: {output_dir}")

## 5. 模型保存位置

### 阿里云绑定目录
- **数据盘**: `/mnt/data` (持久化存储)
- **模型保存**: `/mnt/data/models/{日期}/`
- **自动命名**: 按训练日期自动创建目录

### 目录结构
```
/mnt/data/
├── models/
│   ├── 2026-04-09/          # 按日期保存
│   │   ├── adapter_config.json
│   │   ├── adapter_model.safetensors
│   │   └── training_report.json
│   └── 2026-04-10/
│       └── ...
└── logs/
    └── training.log
```

### 注意事项
- 绑定目录数据不会丢失
- 每次训练自动创建新日期目录
- 可随时回滚到历史版本

In [ ]:
# 检查模型保存结果
import os
from datetime import datetime

# 检查绑定目录
data_dir = "/mnt/data"
if os.path.exists(data_dir):
    print(f"✓ 绑定目录存在: {data_dir}")
    
    # 列出所有已训练的模型
    models_dir = os.path.join(data_dir, "models")
    if os.path.exists(models_dir):
        dates = sorted(os.listdir(models_dir))
        print(f"\n已保存的模型 ({len(dates)}个版本):")
        for date in dates:
            model_path = os.path.join(models_dir, date)
            if os.path.isdir(model_path):
                files = os.listdir(model_path)
                total_size = sum(
                    os.path.getsize(os.path.join(model_path, f)) / 1024 / 1024
                    for f in files if os.path.isfile(os.path.join(model_path, f))
                )
                print(f"  {date}: {len(files)}文件, {total_size:.1f}MB")
    else:
        print("  暂无已保存模型")
else:
    print(f"✗ 绑定目录不存在: {data_dir}")
    print("  请确认已正确配置阿里云数据盘")

## 7. 打包下载

In [ ]:
# 打包最新模型
import os
from datetime import datetime

data_dir = "/mnt/data"
models_dir = os.path.join(data_dir, "models")

if os.path.exists(models_dir):
    # 获取最新日期的模型
    dates = sorted(os.listdir(models_dir), reverse=True)
    if dates:
        latest_date = dates[0]
        model_path = os.path.join(models_dir, latest_date)
        
        # 打包
        tar_file = f"/mnt/data/method-thinker-{latest_date}.tar.gz"
        !cd /mnt/data/models && tar -czvf {tar_file} {latest_date}/
        
        file_size = os.path.getsize(tar_file) / 1024 / 1024
        print(f"✓ 模型已打包: {tar_file}")
        print(f"  文件大小: {file_size:.1f} MB")
    else:
        print("✗ 未找到已训练模型")
else:
    print("✗ 模型目录不存在")

In [ ]:
# 下载方式
print("=" * 50)
print("模型下载方式")
print("=" * 50)

from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")

print("""
方式1: 阿里云控制台下载
  - 进入PAI-DSW实例详情页
  - 点击"数据盘" → 文件管理
  - 下载 method-thinker-{date}.tar.gz

方式2: SCP下载 (需开放SSH)
  scp user@服务器IP:/mnt/data/method-thinker-{date}.tar.gz ./

方式3: OSS上传后下载
  - 运行cell 6的OSS上传代码
  - 从OSS控制台下载

方式4: 阿里云文件管理器
  - 在JupyterLab文件浏览器中
  - 右键点击tar.gz文件 → Download
""".format(date=today))

## 8. 验证模型

In [ ]:
# 测试最新模型
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from datetime import datetime

# 查找最新模型
data_dir = "/mnt/data"
models_dir = os.path.join(data_dir, "models")

if os.path.exists(models_dir):
    dates = sorted(os.listdir(models_dir), reverse=True)
    if dates:
        latest_date = dates[0]
        model_path = os.path.join(models_dir, latest_date)
        
        print(f"加载模型: {model_path}")
        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="auto",
            torch_dtype=torch.float16
        )
        
        # 测试推理
        test_input = "解方程: 2x + 3 = 7"
        inputs = tokenizer(test_input, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256)
        
        print(f"\n问题: {test_input}")
        print(f"回答: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")
        print(f"\n✓ 模型验证成功")
    else:
        print("✗ 未找到已训练模型")
else:
    print("✗ 模型目录不存在，请先运行训练")